In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP

# Quick check on hardware setup
device_count = torch.cuda.device_count()
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()} | Devices found: {device_count}")

PyTorch Version: 2.11.0+cpu
CUDA Available: False | Devices found: 0


In [2]:
class Classifier(nn.Module):
    def __init__(self, in_features=10, out_features=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 32),
            nn.ReLU(),
            nn.Linear(32, out_features)
        )

    def forward(self, x):
        return self.net(x)

def get_dataloader(batch_size=16):
    # Dummy dataset simulating real feature vectors and binary labels
    X = torch.randn(200, 10)
    y = torch.randint(0, 2, (200,))
    dataset = TensorDataset(X, y)

    # DistributedSampler ensures each GPU gets a unique subset of data
    sampler = DistributedSampler(dataset, shuffle=True)
    loader = DataLoader(dataset, batch_size=batch_size, sampler=sampler)
    return loader, sampler

In [3]:
def init_distributed(rank: int, world_size: int, backend: str):
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'
    dist.init_process_group(backend=backend, rank=rank, world_size=world_size)

def run_trainer(rank: int, world_size: int, epochs: int = 5):
    use_cuda = torch.cuda.is_available()
    backend = 'nccl' if use_cuda else 'gloo'

    init_distributed(rank, world_size, backend)

    # Assign target device
    if use_cuda:
        torch.cuda.set_device(rank)
        device = torch.device(f'cuda:{rank}')
    else:
        device = torch.device('cpu')

    # Build model and wrap with DDP
    model = Classifier().to(device)
    ddp_model = DDP(model, device_ids=[rank] if use_cuda else None)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(ddp_model.parameters(), lr=1e-3)

    loader, sampler = get_dataloader(batch_size=16)

    for epoch in range(epochs):
        # Set epoch on sampler for proper multi-GPU shuffling
        sampler.set_epoch(epoch)
        ddp_model.train()
        running_loss = 0.0

        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            outputs = ddp_model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()  # Automatic gradient synchronization across processes
            optimizer.step()

            running_loss += loss.item()

        # Log metrics only from the master process to avoid output clutter
        if rank == 0:
            avg_loss = running_loss / len(loader)
            print(f"Epoch {epoch + 1:02d}/{epochs:02d} | Train Loss: {avg_loss:.4f}")

    dist.destroy_process_group()